# SQL Tutorial: Audit Sample Metadata

**End goal:** create a tiny SQLite database, check sample metadata, and join candidate genes to annotations.

SQL is a great way to make sample labels, joins, and missing values explicit before analysis.

## Setup: load the tutorial data

Run this cell first. It uses the local files when they already exist and downloads the tiny tutorial data when you open the notebook in Colab or another fresh environment.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)
raw_base = 'https://raw.githubusercontent.com/Caffeinated-Code/Bioinformatics-Field-Guide/main/content/resources/week-02/data'
for file_name in ['samples.tsv', 'mini_transcripts.fasta', 'gene_counts.tsv', 'gene_annotations.tsv']:
    target = data_dir / file_name
    if not target.exists():
        urlretrieve(f"{raw_base}/{file_name}", target)

sorted(path.name for path in data_dir.iterdir())


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

data_dir = Path('../data')
con = sqlite3.connect(':memory:')

pd.read_csv(data_dir / 'samples.tsv', sep='\t').to_sql('samples', con, index=False, if_exists='replace')
pd.read_csv(data_dir / 'gene_annotations.tsv', sep='\t').to_sql('gene_annotations', con, index=False, if_exists='replace')

def sql(query):
    return pd.read_sql_query(query, con)

sql('SELECT * FROM samples')

## 1. Count samples by condition

This should be one of your first metadata checks.

In [ ]:
sql('''
SELECT condition, COUNT(*) AS n_samples
FROM samples
GROUP BY condition
ORDER BY condition;
''')

## 2. Check batch balance

A batch that perfectly matches condition can confuse biological interpretation.

In [ ]:
sql('''
SELECT batch, condition, COUNT(*) AS n_samples
FROM samples
GROUP BY batch, condition
ORDER BY batch, condition;
''')

## 3. Look for duplicated sample IDs

A duplicate sample identifier can break joins or silently duplicate rows.

In [ ]:
sql('''
SELECT sample_id, COUNT(*) AS n
FROM samples
GROUP BY sample_id
HAVING COUNT(*) > 1;
''')

## 4. Join candidate genes to annotations

Create a tiny candidate table and join it to gene symbols and pathway labels.

In [ ]:
candidate_genes = pd.DataFrame({
    'gene_id': ['ENSG00000141510', 'ENSG00000157764', 'ENSG_NOT_FOUND'],
    'reason': ['stress response', 'signal change', 'example missing annotation']
})
candidate_genes.to_sql('candidate_genes', con, index=False, if_exists='replace')

sql('''
SELECT c.gene_id, a.symbol, a.pathway, c.reason
FROM candidate_genes c
LEFT JOIN gene_annotations a
  ON c.gene_id = a.gene_id
ORDER BY c.gene_id;
''')

## Your turn

- Filter samples to `reads_million >= 23`.
- Count samples by `tissue`.
- Find candidate genes with missing annotations by adding `WHERE a.symbol IS NULL` to the join.

**Confidence check:** you should be able to explain why SQL joins are useful before running an omics model.